In [1]:
import os
import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from lib.model.resNet.resNet import ResNet
from tqdm import tqdm
from configs.train import parser



In [2]:
args = parser.parse_args(args=[])


In [3]:
if args.use_tensorboard:
    log_dir = os.path.join(args.output_dir, args.tensorboard_dir, str(args.session))
    if not os.path.exists(log_dir):
        os.makedirs(log_dir)
    writer = SummaryWriter(log_dir=log_dir)

In [4]:
def get_fashion_mnist_labels(labels):
    """return text-labels of Fashion-mnist dataset"""
    text_labels = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat', 
                  'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
    return [text_labels[int(i)] for i in labels]

# pre-process
# type: PIL -> torch.float32.Tensor
# pixel value in [0, 1]
# resize to (256, 256)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256))])

mnist_train = datasets.FashionMNIST(
    root="./data", train=True, transform=transform, download=False
)
# mnist_test = datasets.FashionMNIST(
#     root="./data", train=False, transform=trans, download=False
# )

# mnist_train: (tuple0, tuple1, ...)
# tuplei: (image, label)
# image: shape=(C, H, W)
trainSize = len(mnist_train)
print("training size: {}".format(trainSize))
# print("test size: {}".format((len(mnist_test))))
print("sample 0 image shape: {}".format(mnist_train[0][0].shape))
print("sample 0 label format: {}".format(mnist_train[0][1]))


DataLoaderTrain = DataLoader(mnist_train, batch_size=args.batchSize,
                            shuffle=args.shuffle, num_workers=args.num_workers)

training size: 60000
sample 0 image shape: torch.Size([1, 256, 256])
sample 0 label format: 9


In [5]:
resNet = ResNet(pretrained=args.pretrained)
resNet.initModule()
resNet.train()
resNet.to(args.device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resNet.parameters(), lr=args.lr)


>>> Initializing ResNet model.
>>> conv1 weight shape: (3, 64) - > (1, 64)
>>> fc weight shape: (512, 1000) - > (512, 10)


In [ ]:
iters_per_epoch = int(trainSize / args.batchSize)

for epoch in tqdm(range(args.num_epochs), desc="Epochs", position=0):
# for epoch in range(args.num_epochs):
    if args.use_tensorboard:
        loss_avg_temp = 0.0
    for step, (images, label) in tqdm(enumerate(DataLoaderTrain), total=len(DataLoaderTrain)):
        print(step)
        # move data to device
        images = images.to(args.device)
        label = label.to(args.device)
        # train
        logits = resNet(images)
        loss = criterion(logits, label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        loss_avg_temp += loss.item()
        print(loss)
    # log
    if args.use_tensorboard and ((step+1) % args.log_interval == 0):
        # compute train accuracy
        with torch.no_grad():
            train_acc = (logits.argmax(dim=1) == label).float().mean().item()
        # loss average
        loss_avg_temp /= args.log_interval
        writer.add_scalar('train/loss', loss_avg_temp, epoch * iters_per_epoch + step)
        writer.add_scalar('train/accuracy', train_acc, epoch * iters_per_epoch + step)
        loss_temp = 0
        
    # save model checkpoint
    if (epoch+1) % args.save_interval == 0:
        if os.path.exists(args.checkpoint_dir) is False:
            os.makedirs(args.checkpoint_dir)
        modelPath = os.path.join(args.checkpoint_dir, 
                                 "resNet_{}_{}_{}.pth".format(args.session, epoch+1, step+1))
        checkpoint = {'epoch': epoch + 1,
                      'step': step + 1,
                      'model_state_dict': resNet.state_dict(),
                      'optimizer_state_dict': optimizer.state_dict(),
                      "loss": loss.item()}


Epochs:   0%|          | 0/10 [00:00<?, ?it/s]